# Coordinate Reference Systems (CRS) — Sinú Basin Edition

Companion notebook for the **EcoGeo Tutor** tutorial: *Coordinate Reference Systems (CRS)*.

Adapted to use the Sinú basin shapefile and DEM from the earlier tutorial, instead of
a generic placeholder file — same concepts (geographic vs projected vs web CRS,
reprojection, distance/area calculation, and common CRS mistakes), applied to real data
already on disk.


## Setup

In [2]:
import geopandas as gpd
import rasterio
from shapely.geometry import Point
from pathlib import Path
import folium

data      = Path(r"C:\Users\Jhonnatan\Downloads\hybas_sa_lev01-12_v1c")
basin_path = data / "sinu_basin.shp"
dem_path   = data / "dem_30.tif"

# Sanity check before opening anything — catches path typos immediately
print(basin_path, "->", basin_path.exists())
print(dem_path,   "->", dem_path.exists())

C:\Users\Jhonnatan\Downloads\hybas_sa_lev01-12_v1c\sinu_basin.shp -> True
C:\Users\Jhonnatan\Downloads\hybas_sa_lev01-12_v1c\dem_30.tif -> True


## 1. Geographic CRS — EPSG:4326 (WGS84)

Raw storage format: degrees, not metres. Never calculate area directly in this CRS.

In [3]:
gdf = gpd.read_file(basin_path)

print(gdf.crs)
print(gdf.crs.axis_info)

centroid = gdf.geometry.iloc[0].centroid
print("Centroid:", centroid)   # in DEGREES — this is the Sinú basin's actual centroid

# DO NOT calculate area in geographic CRS!
# gdf.geometry.area  <- returns degrees^2, meaningless

EPSG:4326
[Axis(name=Geodetic latitude, abbrev=Lat, direction=north, unit_auth_code=EPSG, unit_code=9122, unit_name=degree), Axis(name=Geodetic longitude, abbrev=Lon, direction=east, unit_auth_code=EPSG, unit_code=9122, unit_name=degree)]
Centroid: POINT (-75.98483059157228 8.338664693050188)


## 2. Projected CRS — EPSG:32618 (UTM Zone 18N)

Metres-based; accurate for area and distance calculations within its zone.
UTM 18N covers northern Colombia, including the full Sinú basin.

In [4]:
gdf_utm = gdf.to_crs("EPSG:32618")
print(gdf_utm.crs)

# Now area is meaningful — compare against the SUB_AREA field from HydroBASINS
gdf_utm["area_km2"] = gdf_utm.geometry.area / 1e6
print(f"Computed area: {gdf_utm.iloc[0]['area_km2']:.1f} km2")
print(f"HydroBASINS SUB_AREA field: {gdf.iloc[0]['SUB_AREA']:.1f} km2")
# These should be very close — a good cross-check that the reprojection is correct

EPSG:32618
Computed area: 13999.4 km2
HydroBASINS SUB_AREA field: 14065.1 km2


Distance between two points, correctly measured in metres after reprojecting:

In [5]:
p1 = gpd.GeoSeries([Point(-75.9, 8.5)], crs="EPSG:4326").to_crs("EPSG:32618")
p2 = gpd.GeoSeries([Point(-75.5, 9.0)], crs="EPSG:4326").to_crs("EPSG:32618")

dist_km = p1.distance(p2).iloc[0] / 1000
print(f"Distance: {dist_km:.2f} km")

Distance: 70.65 km


Render the basin on an interactive web map — note the CRS switches back to EPSG:4326 here, since that's what web map tiles expect:

In [6]:
m = folium.Map(location=[8.5, -75.8], zoom_start=8)
folium.GeoJson(gdf.to_crs("EPSG:4326")).add_to(m)
m.save("sinu_basin.html")
print("Saved: sinu_basin.html")

Saved: sinu_basin.html


## 3. Web Mercator — EPSG:3857

For rendering map tiles only. Never use for area or distance analysis — severe
distortion at high latitudes (though at the Sinú basin's low latitude, ~8-9°N,
the distortion is much milder than it would be near the poles — still not worth
relying on for anything quantitative).

In [7]:
gdf_webmercator = gdf.to_crs("EPSG:3857")
print(gdf_webmercator.crs)

# WRONG - do not calculate area in EPSG:3857
# gdf_webmercator.geometry.area   <- distorted, don't use this number

# Compare: EPSG:3857 area vs the correct UTM area computed above
wrong_area_km2 = gdf_webmercator.geometry.area.iloc[0] / 1e6
print(f"Web Mercator 'area' (wrong):  {wrong_area_km2:.1f} km2")
print(f"UTM 18N area (correct):       {gdf_utm.iloc[0]['area_km2']:.1f} km2")
print(f"HydroBASINS SUB_AREA field:   {gdf.iloc[0]['SUB_AREA']:.1f} km2")

EPSG:3857
Web Mercator 'area' (wrong):  14401.1 km2
UTM 18N area (correct):       13999.4 km2
HydroBASINS SUB_AREA field:   14065.1 km2


## 4. Check the DEM's CRS too

CRS mismatches aren't just a vector problem — always check the raster's CRS
before combining it with a vector layer.

In [8]:
with rasterio.open(dem_path) as src:
    print("DEM CRS:   ", src.crs)
    print("DEM bounds:", src.bounds)

print("Basin CRS: ", gdf.crs)
# Both EPSG:4326 here - they already match, so no reprojection is needed
# before operations like rasterio.mask.mask() from the previous notebook

DEM CRS:    EPSG:4326
DEM bounds: BoundingBox(left=-76.70041666669015, bottom=7.000416666678713, right=-75.25041666669047, top=9.600416666678122)
Basin CRS:  EPSG:4326


## 5. The three most common CRS mistakes

Using the same Sinú files to demonstrate each one concretely.

**Mistake 1 — Layer misalignment.** Mixing CRS without checking before a spatial operation:

In [9]:
# WRONG (illustrative - would misalign if basin and DEM had different CRS):
# overlay = gpd.overlay(some_other_layer, gdf)   # fails silently if CRS differ

# RIGHT: always align CRS before any spatial operation
other_crs_layer = gdf.to_crs("EPSG:32618")            # pretend this came from elsewhere
aligned         = other_crs_layer.to_crs(gdf.crs)      # reproject to match before combining
print("Aligned CRS matches basin CRS:", aligned.crs == gdf.crs)

Aligned CRS matches basin CRS: True


**Mistake 2 — Area calculated in degrees.** Already demonstrated above, repeated here as the core lesson:

In [10]:
# WRONG:
# gdf.geometry.area   <- returns degrees^2, meaningless

# RIGHT: reproject to a metres-based CRS first
area_km2_correct = gdf.to_crs("EPSG:32618").geometry.area.iloc[0] / 1e6
print(f"Correct area: {area_km2_correct:.1f} km2")

Correct area: 13999.4 km2


**Mistake 3 — Unknown or missing CRS.** What to do if a file has no CRS assigned:

In [11]:
# Simulate a file with no CRS
no_crs_gdf = gdf.copy()
no_crs_gdf.crs = None
print("CRS before fix:", no_crs_gdf.crs)

# WRONG: to_crs() on data with no CRS raises an error - there's nothing to convert FROM
# no_crs_gdf.to_crs("EPSG:32618")   # would fail

# RIGHT: assign the known CRS first (does NOT move coordinates), THEN reproject
no_crs_gdf = no_crs_gdf.set_crs("EPSG:4326")
no_crs_gdf = no_crs_gdf.to_crs("EPSG:32618")
print("CRS after fix: ", no_crs_gdf.crs)

CRS before fix: None
CRS after fix:  EPSG:32618


## Reference — common EPSG codes

| EPSG Code | Name | Type | Coverage | Best use |
|---|---|---|---|---|
| EPSG:4326 | WGS 84 | Geographic | Global | GPS, raw data, data exchange |
| EPSG:3857 | Web Mercator | Projected (spherical) | Global (web) | Web map tiles only |
| EPSG:32618 | UTM Zone 18N | Projected (UTM) | Colombia / Caribbean | Analysis in N Colombia (used throughout this notebook) |
| EPSG:32633 | UTM Zone 33N | Projected (UTM) | Italy (centre) | Analysis in Italy |
| EPSG:9377 | MAGNA-SIRGAS / CTM-12 | Projected | Colombia | Official Colombian CRS |

Look up any EPSG code at [epsg.io](https://epsg.io).

---
*Adapted from the EcoGeo Tutor CRS tutorial, using the Sinú basin dataset.*
